# Interactive Point Cloud Visualization

This notebook reads EvoEngine's generated `.ply` and visualizes all points.

- Preferred backend: `plotly` (interactive rotate / zoom / pan)
- Fallback backend: `matplotlib`
- Coordinate remap for visualization: EvoEngine uses `Y-up`, while the plot uses the third axis as the vertical axis, so we display points as `(x, z, y)`

Color views:
- `instance_index`
- `leaf_index`


In [135]:
from pathlib import Path
import numpy as np

try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

import matplotlib.pyplot as plt

ply_path = Path(r"C:\Users\sdjkl\CG\VRBioTalk\test_scan.ply")
point_size = 2
backend = "plotly" if HAS_PLOTLY else "matplotlib"
print("backend:", backend)
ply_path

backend: plotly


WindowsPath('C:/Users/sdjkl/CG/VRBioTalk/test_scan.ply')

In [136]:
PLY_TYPE_TO_DTYPE = {
    "char": np.int8,
    "int8": np.int8,
    "uchar": np.uint8,
    "uint8": np.uint8,
    "short": np.int16,
    "int16": np.int16,
    "ushort": np.uint16,
    "uint16": np.uint16,
    "int": np.int32,
    "int32": np.int32,
    "uint": np.uint32,
    "uint32": np.uint32,
    "float": np.float32,
    "float32": np.float32,
    "double": np.float64,
    "float64": np.float64,
}

def parse_ply_header(path: Path):
    header_lines = []
    with path.open("rb") as f:
        while True:
            line = f.readline()
            if not line:
                raise ValueError("Unexpected EOF while reading PLY header")
            decoded = line.decode("ascii").rstrip("\r\n")
            header_lines.append(decoded)
            if decoded == "end_header":
                data_start = f.tell()
                break

    fmt = None
    elements = []
    current_element = None

    for line in header_lines:
        parts = line.split()
        if not parts:
            continue
        if parts[0] == "format":
            fmt = parts[1]
        elif parts[0] == "element":
            current_element = {
                "name": parts[1],
                "count": int(parts[2]),
                "properties": [],
            }
            elements.append(current_element)
        elif parts[0] == "property":
            if current_element is None:
                raise ValueError("Property found before any element")
            if parts[1] == "list":
                raise NotImplementedError("List properties are not supported in this notebook")
            current_element["properties"].append((parts[2], parts[1]))

    return fmt, elements, data_start, header_lines

def read_evoengine_ply(path: Path):
    fmt, elements, data_start, header_lines = parse_ply_header(path)
    if fmt != "binary_little_endian":
        raise NotImplementedError(f"Unsupported PLY format: {fmt}")

    data = {}
    with path.open("rb") as f:
        f.seek(data_start)
        for element in elements:
            dtype = np.dtype([(name, PLY_TYPE_TO_DTYPE[typ]) for name, typ in element["properties"]])
            data[element["name"]] = np.fromfile(f, dtype=dtype, count=element["count"])
    return data, header_lines

ply_data, header_lines = read_evoengine_ply(ply_path)
print("Elements:", list(ply_data.keys()))
for key, value in ply_data.items():
    print(key, value.dtype.names, len(value))

Elements: ['vertex', 'type_index', 'instance_index', 'leaf_index']
vertex ('x', 'y', 'z') 40045
type_index ('type_index',) 40045
instance_index ('instance_index',) 40045
leaf_index ('leaf_index',) 40045


In [137]:
vertex = ply_data["vertex"]
xyz_engine = np.column_stack([vertex["x"], vertex["y"], vertex["z"]]).astype(np.float32)
# Remap for visualization: engine (x, y, z) -> plot (x, z, y)
xyz_plot = xyz_engine[:, [0, 2, 1]]
xyz_plot[:, 2] *= -1.0  # Flip y for better visualization

instance_index = ply_data["instance_index"]["instance_index"].astype(np.int32) if "instance_index" in ply_data else None
leaf_index = ply_data["leaf_index"]["leaf_index"].astype(np.int32) if "leaf_index" in ply_data else None
type_index = ply_data["type_index"]["type_index"].astype(np.int32) if "type_index" in ply_data else None

print("engine xyz shape:", xyz_engine.shape)
print("plot xyz shape:", xyz_plot.shape)
if instance_index is not None:
    print("instance unique:", np.unique(instance_index).tolist())
if leaf_index is not None:
    print("leaf unique:", np.unique(leaf_index).tolist()[:40])
if type_index is not None:
    print("type unique:", np.unique(type_index).tolist())

engine xyz shape: (40045, 3)
plot xyz shape: (40045, 3)
instance unique: [0, 1, 2, 3, 4]
leaf unique: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
type unique: [0]


In [138]:
def label_colors_rgba(labels, invalid_value=-1, cmap_name="tab20"):
    labels = np.asarray(labels)
    colors = np.zeros((len(labels), 4), dtype=np.float32)
    valid_mask = labels != invalid_value
    colors[~valid_mask] = np.array([0.6, 0.6, 0.6, 1.0], dtype=np.float32)
    unique_valid = np.unique(labels[valid_mask])
    cmap = plt.get_cmap(cmap_name, max(len(unique_valid), 1))
    lut = {label: cmap(i) for i, label in enumerate(unique_valid)}
    for i, label in enumerate(labels):
        if label in lut:
            colors[i] = lut[label]
    return colors

def rgba_to_plotly(colors_rgba):
    rgb = np.clip(colors_rgba[:, :3] * 255.0, 0, 255).astype(np.uint8)
    return [f"rgb({r},{g},{b})" for r, g, b in rgb]

def plot_point_cloud(labels, title, invalid_value=-1, cmap_name="tab20"):
    colors_rgba = label_colors_rgba(labels, invalid_value=invalid_value, cmap_name=cmap_name)
    if backend == "plotly":
        colors = rgba_to_plotly(colors_rgba)
        fig = go.Figure(
            data=[
                go.Scatter3d(
                    x=xyz_plot[:, 0],
                    y=xyz_plot[:, 1],
                    z=xyz_plot[:, 2],
                    mode="markers",
                    marker=dict(size=point_size, color=colors, opacity=0.9),
                )
            ]
        )
        fig.update_layout(
            title=title,
            scene=dict(
                aspectmode="data",
                xaxis_title="X",
                yaxis_title="Z",
                zaxis_title="Y (up)",
            ),
            margin=dict(l=0, r=0, b=0, t=40),
        )
        fig.show()
    else:
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection="3d")
        ax.scatter(xyz_plot[:, 0], xyz_plot[:, 1], xyz_plot[:, 2], c=colors_rgba, s=point_size, linewidths=0)
        ax.set_title(title)
        ax.set_xlabel("X")
        ax.set_ylabel("Z")
        ax.set_zlabel("Y (up)")
        span = np.ptp(xyz_plot, axis=0)
        span[span == 0] = 1.0
        ax.set_box_aspect(span)
        plt.show()

In [139]:
if instance_index is None:
    print("No instance_index element found in this PLY.")
else:
    plot_point_cloud(instance_index, "Point Cloud Colored by instance_index")

In [140]:
if leaf_index is None:
    print("No leaf_index element found in this PLY.")
else:
    plot_point_cloud(leaf_index, "Point Cloud Colored by leaf_index")